<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/playfair_cipher_5x5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playfair Cipher

## History
The Playfair Cipher was invented by Charles Wheatstone in 1854, but it is named after Lord Playfair, who promoted its use to the British government. It was actually used by the British army in the Boer War and in World War 1, because it was fast enough to use by hand in the field, and much stronger than a simple substitution cipher.

## What is Playfair Cipher?
Playfair Cipher is different from every cipher we have looked at so far, because it does not encrypt one letter at a time. It encrypts **pairs of letters**, called **digraphs**. This makes simple frequency analysis on single letters useless against it, because the cipher is not hiding the frequency of A, B, C individually, it is hiding the frequency of letter pairs.

The whole cipher is built around a **5x5 grid of letters**, called the **key square**. Since the English alphabet has 26 letters and the grid only has 25 cells, the letters **I and J share the same cell**. Wherever a J appears in the message, it is treated as an I.

## Cryptography Algorithm

### 1. Building the Key Square
1.  Pick a keyword, for example **MONARCHY**.
2.  Write the keyword into the grid, left to right, top to bottom, skipping any letter that has already been used.
3.  Fill in the rest of the grid with the remaining letters of the alphabet, in order, also skipping repeats. Remember, I and J share one cell.

For the keyword **MONARCHY**, the key square looks like this:

| | | | | |
|---|---|---|---|---|
| M | O | N | A | R |
| C | H | Y | B | D |
| E | F | G | I/J | K |
| L | P | Q | S | T |
| U | V | W | X | Z |

### 2. Preparing the Plaintext
Before encryption, the plaintext must be cleaned and split into pairs, using these rules:

1.  Remove all spaces, numbers, and punctuation. Keep only letters.
2.  Replace every J with an I.
3.  Split the letters into pairs, from left to right.
4.  If both letters in a pair are the same, insert an **X** between them, and re-pair the rest of the message.
5.  If the very last letter is left alone with no partner, pad it with an **X**.

For example, **HELLO** becomes **HE LX LO**, because the double L needs an X inserted between them.

### 3. Encryption Rules
Every pair of letters is located inside the key square, and one of three rules is applied, depending on where the two letters sit relative to each other.

**Rule 1: Same Row**
If both letters are in the same row, replace each letter with the letter immediately to its right, wrapping around to the start of the row if needed.

**Rule 2: Same Column**
If both letters are in the same column, replace each letter with the letter immediately below it, wrapping around to the top of the column if needed.

**Rule 3: Rectangle**
If the letters are in different rows and different columns, they form the two opposite corners of a rectangle. Replace each letter with the letter that sits in its own row, but in the column of the other letter.

### 4. Decryption Rules
Decryption uses the exact same key square and the exact same three rules, just reversed.

*   **Same Row**: move each letter one step to the **left** instead of right.
*   **Same Column**: move each letter one step **up** instead of down.
*   **Rectangle**: exactly the same rule as encryption, since swapping columns twice brings you back to where you started.

### 5. Fully Worked Example (By Hand)
Using the key square built from **MONARCHY** above.

**Example A: Same Row rule.** Encrypt the pair **MO**.
Both M and O sit in row 0, at columns 0 and 1. Moving each one step to the right:
*   M (row 0, col 0) becomes the letter at (row 0, col 1) = **O**
*   O (row 0, col 1) becomes the letter at (row 0, col 2) = **N**

So **MO encrypts to ON**.

**Example B: Same Column rule.** Encrypt the pair **MC**.
Both M and C sit in column 0, at rows 0 and 1. Moving each one step down:
*   M (row 0, col 0) becomes the letter at (row 1, col 0) = **C**
*   C (row 1, col 0) becomes the letter at (row 2, col 0) = **E**

So **MC encrypts to CE**.

**Example C: Rectangle rule.** Encrypt the pair **HE**.
H sits at row 1, col 1. E sits at row 2, col 0. Different row, different column, so this is a rectangle.
*   H stays in row 1, but moves to E's column (col 0), giving the letter at (row 1, col 0) = **C**
*   E stays in row 2, but moves to H's column (col 1), giving the letter at (row 2, col 1) = **F**

So **HE encrypts to CF**.

This is exactly the logic the code below runs automatically for every pair in a full message.

### 1. Import Dependencies

In [1]:
import random

### 2. Build the Key Square

In [2]:
def build_key_square(keyword: str) -> list:
    keyword = keyword.upper().replace("J", "I")

    seen = set()
    square = []

    # first place every unique letter of the keyword
    for ch in keyword:
        if ch.isalpha() and ch not in seen:
            seen.add(ch)
            square.append(ch)

    # then fill in the rest of the alphabet, skipping J since I/J share a cell
    for ch in "ABCDEFGHIKLMNOPQRSTUVWXYZ":
        if ch not in seen:
            seen.add(ch)
            square.append(ch)

    grid = [square[i * 5:(i + 1) * 5] for i in range(5)]
    return grid

def find_position(grid: list, ch: str):
    for row in range(5):
        for col in range(5):
            if grid[row][col] == ch:
                return row, col
    return None

### 3. Prepare the Plaintext into Digraphs

In [3]:
def prepare_text(text: str) -> list:
    # keep only letters, and merge J into I, since Playfair has no separate J cell
    cleaned = "".join(ch for ch in text.upper() if ch.isalpha()).replace("J", "I")

    digraphs = []
    i = 0
    while i < len(cleaned):
        first = cleaned[i]

        if i + 1 < len(cleaned):
            second = cleaned[i + 1]
            if first == second:
                # double letters need an X inserted between them
                digraphs.append(first + "X")
                i += 1
            else:
                digraphs.append(first + second)
                i += 2
        else:
            # last letter with no partner gets padded with X
            digraphs.append(first + "X")
            i += 1

    return digraphs

### 4. Generate a Random Key

In [4]:
def generate_random_key() -> str:
    # shuffle all 25 letters (no J) into a random order to build a random key square
    letters = list("ABCDEFGHIKLMNOPQRSTUVWXYZ")
    random.shuffle(letters)
    return "".join(letters)

### 5. Encryption

In [5]:
def encrypt_pair(grid: list, a: str, b: str) -> str:
    row_a, col_a = find_position(grid, a)
    row_b, col_b = find_position(grid, b)

    if row_a == row_b:
        # same row: shift right, wrap around with modulo 5
        return grid[row_a][(col_a + 1) % 5] + grid[row_b][(col_b + 1) % 5]
    elif col_a == col_b:
        # same column: shift down, wrap around with modulo 5
        return grid[(row_a + 1) % 5][col_a] + grid[(row_b + 1) % 5][col_b]
    else:
        # rectangle: swap columns, keep each letter's own row
        return grid[row_a][col_b] + grid[row_b][col_a]

def encrypt(text: str, keyword: str) -> str:
    grid = build_key_square(keyword)
    digraphs = prepare_text(text)
    return "".join(encrypt_pair(grid, pair[0], pair[1]) for pair in digraphs)

### 6. Decryption

In [6]:
def decrypt_pair(grid: list, a: str, b: str) -> str:
    row_a, col_a = find_position(grid, a)
    row_b, col_b = find_position(grid, b)

    if row_a == row_b:
        # same row: shift left, wrap around with modulo 5
        return grid[row_a][(col_a - 1) % 5] + grid[row_b][(col_b - 1) % 5]
    elif col_a == col_b:
        # same column: shift up, wrap around with modulo 5
        return grid[(row_a - 1) % 5][col_a] + grid[(row_b - 1) % 5][col_b]
    else:
        # rectangle: same swap as encryption
        return grid[row_a][col_b] + grid[row_b][col_a]

def decrypt(cipher_text: str, keyword: str) -> str:
    grid = build_key_square(keyword)
    pairs = [cipher_text[i:i + 2] for i in range(0, len(cipher_text), 2)]
    return "".join(decrypt_pair(grid, pair[0], pair[1]) for pair in pairs)

### 7. Verify the Hand Worked Example in Code

In [7]:
hand_keyword = "MONARCHY"
hand_grid = build_key_square(hand_keyword)

print("Key Square:")
for row in hand_grid:
    print(row)

print("\nMO ->", encrypt_pair(hand_grid, "M", "O"), "(should match ON from the hand example)")
print("MC ->", encrypt_pair(hand_grid, "M", "C"), "(should match CE from the hand example)")
print("HE ->", encrypt_pair(hand_grid, "H", "E"), "(should match CF from the hand example)")

Key Square:
['M', 'O', 'N', 'A', 'R']
['C', 'H', 'Y', 'B', 'D']
['E', 'F', 'G', 'I', 'K']
['L', 'P', 'Q', 'S', 'T']
['U', 'V', 'W', 'X', 'Z']

MO -> ON (should match ON from the hand example)
MC -> CE (should match CE from the hand example)
HE -> CF (should match CF from the hand example)


### 8. Example usage

In [8]:
key = generate_random_key()
print(f"Generated Random Key: {key}")

Generated Random Key: BEPIOXTMNVGZLUSYQFHRAWKCD


In [9]:
plaintext = "TOP secret Massage! Agent Roy, visit Area fifty-one"
cleaned_digraph_text = "".join(prepare_text(plaintext))
print(f"Original Plain Text: {plaintext}")
print(f"Cleaned Plain Text: {cleaned_digraph_text}")
print(f"Prepared Digraphs: {prepare_text(plaintext)}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

# cleaned_digraph_text = "".join(prepare_text(plaintext))
match = cleaned_digraph_text == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent Roy, visit Area fifty-one
Cleaned Plain Text: TOPSECRETMASSAGEAGENTROYVISITAREAFIFTYONEX
Prepared Digraphs: ['TO', 'PS', 'EC', 'RE', 'TM', 'AS', 'SA', 'GE', 'AG', 'EN', 'TR', 'OY', 'VI', 'SI', 'TA', 'RE', 'AF', 'IF', 'TY', 'ON', 'EX']
Encrypted: VEOLIWQOMNDGGDZBBYITVQBRNOUOXWQOKYPHXQIVBT
Decrypted: TOPSECRETMASSAGEAGENTROYVISITAREAFIFTYONEX
Verification Match:True
